In [1]:
# %pip install pypdf tqdm

In [2]:
import os
from tqdm import tqdm
from pypdf import PdfReader

def extract_pdfs_recursively(folder_path, output_txt_path):
    """
    Finds all PDF files in folder_path and all of its subfolders, 
    extracts their text, and saves them into a single text file.
    """
    # 1. Recursively find all PDF files across all subfolders
    pdf_files = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith('.pdf'):
                # Store the full path to the file
                pdf_files.append(os.path.join(root, file))
    
    if not pdf_files:
        print(f"No PDF files found in '{folder_path}' or its subfolders.")
        return

    print(f"Found {len(pdf_files)} PDF files across all folders. Starting extraction...")

    # 2. Open the single output text file
    with open(output_txt_path, 'w', encoding='utf-8') as outfile:
        
        # 3. Iterate through files with a tqdm progress bar
        for index, file_path in enumerate(tqdm(pdf_files, desc="Processing PDFs")):
            try:
                # Initialize PDF reader
                reader = PdfReader(file_path)
                pdf_text = []
                
                # Extract text page by page
                for page in reader.pages:
                    text = page.extract_text()
                    if text:
                        pdf_text.append(text)
                
                # Join all pages of the current PDF
                full_pdf_text = "\n".join(pdf_text)
                
                # [Optional] Uncomment the line below if you want to know which file the text came from:
                # outfile.write(f"--- START OF FILE: {os.path.basename(file_path)} ---\n")
                
                outfile.write(full_pdf_text)
                
                # 4. Add the "---" separator after each PDF content
                if index < len(pdf_files) - 1:
                    outfile.write("\n\n---\n\n")
                    
            except Exception as e:
                print(f"\nError processing file '{file_path}': {e}")
                
    print(f"\nSuccessfully combined all PDFs into: {output_txt_path}")

In [3]:
# --- Example Usage ---
if __name__ == "__main__":
    # This will now scan this folder AND all folders inside it
    target_folder = "./Docs" 
    output_file = "./combined_output.txt"
    
    # Run the function
    extract_pdfs_recursively(target_folder, output_file)

Found 4714 PDF files across all folders. Starting extraction...


Processing PDFs: 100%|██████████| 4714/4714 [39:18<00:00,  2.00it/s]  


Successfully combined all PDFs into: ./combined_output.txt


In [6]:
import re

def clean_community_data(text):
    # Phrases commonly found in UI text that we want to discard
    boilerplate_blacklist = [
        "comments",
        "be the first to write a comment",
        "search",
        "platform community",
        "faq submit a request",
        "english (united states)",
        "sort by",
        "worldquant brain",
        "getting started with research",
        "Hello, Community!",
        "brain®",
        "brain tips",
        "n ew post"
    ]
    
    cleaned_lines = []
    
    # Split text line by line
    for line in text.split('\n'):
        line_stripped = line.strip()
        
        # 1. Skip completely empty lines
        if not line_stripped:
            continue
            
        # 2. Skip lines that are just numbers (like standalone 0 or 18)
        if line_stripped.isdigit():
            continue
            
        # 3. Skip UI text matching "Follow 1", "Follow 11", etc.
        if re.match(r'^follow\s+\d+', line_stripped, re.IGNORECASE):
            continue
            
        # 4. Skip timestamps like "3 months ago", "4 days ago"
        if re.search(r'\d+\s+(month|day|week|year)s?\s+ago', line_stripped, re.IGNORECASE):
            continue
        
        # 5. Remove lines that are just a string like "JG63643" (two capitals first and then five numbers)
        if re.match(r'^[A-Z]{2}\d{5}$', line_stripped):
            continue
        
        # 6. Skip lines that contain any of our blacklisted boilerplate phrases
        should_skip = False
        for phrase in boilerplate_blacklist:
            if phrase in line_stripped.lower():
                should_skip = True
                break
                
        if should_skip:
            continue
            
        # If it passes all filters, save the cleaned line
        cleaned_lines.append(line_stripped)

    # Join the lines back together
    full_output = "\n".join(cleaned_lines)

    # Join everything back together
    return re.sub(r'\n\s*\n+', '\n', full_output).strip()

In [5]:
# NEW LOGIC BELOW: Obtain the file content, apply the function, and save as a new file
cleaned_output_file = "./combined_cleaned_output.txt"

if os.path.exists(output_file):
    print("\n--- Starting Text Cleaning Process ---")
    
    # Read the generated file content
    with open(output_file, 'r', encoding='utf-8') as f:
        raw_content = f.read()
        
    print("Applying data cleaning rules...")
    # Apply your cleaning function
    cleaned_content = clean_community_data(raw_content)
    
    # Save as a new file
    with open(cleaned_output_file, 'w', encoding='utf-8') as f:
        f.write(cleaned_content)
        
    print(f"Successfully saved cleaned data to: {cleaned_output_file}")


--- Starting Text Cleaning Process ---
Applying data cleaning rules...
Successfully saved cleaned data to: ./combined_cleaned_output.txt
